# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [1]:
# TODO


## 2. Ressources (psutil)

In [8]:
"""Volet ressources — mesures psutil sur le modèle legacy (RSS, temps, taille, débit)."""
import platform, statistics as stats, subprocess, sys, time
from pathlib import Path

import joblib
import pandas as pd
import psutil
import sklearn
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL = ROOT / "legacy" / "dms_predictor_v1.joblib"
DATA = ROOT / "data" / "dms_dataset.csv"

df = pd.read_csv(DATA)
X = df[["age", "nb_comorbidites", "imc"]].copy()
X["sexe_bin"] = (df["sexe"] == "M").astype(int)
y = df["sejour_prolonge"]

BANC = (f"{platform.system()} {platform.machine()} | Python {platform.python_version()} | "
        f"sklearn {sklearn.__version__} | {psutil.cpu_count(logical=False)} cœurs physiques / "
        f"{psutil.cpu_count()} logiques | {psutil.virtual_memory().total / 1e9:.1f} Go RAM")
print(BANC)

# --- RSS isolé : le kernel a déjà tout importé, on mesure donc dans un process neuf ---
# Le prélude reproduit les imports de predict.py ; sklearn n'y figure pas : c'est joblib.load qui le tire.
PROBE = """
import os, sys, time, psutil, joblib, pandas
{prelude}
p = psutil.Process(os.getpid())
base = p.memory_info().rss
t0 = time.perf_counter(); m = joblib.load(sys.argv[1]); t = time.perf_counter() - t0
print(base, p.memory_info().rss, t)
"""
SKL = "import sklearn.ensemble, sklearn.linear_model"


def probe(path, prelude="", n=5):
    """RSS avant/après joblib.load dans un process neuf ; médiane sur n runs (le 1er paie le cache disque)."""
    rows = []
    for _ in range(n):
        out = subprocess.run([sys.executable, "-c", PROBE.format(prelude=prelude), str(path)],
                             capture_output=True, text=True, cwd=ROOT, check=True)
        b, a, t = out.stdout.split()
        rows.append((int(b) / 1e6, int(a) / 1e6, float(t) * 1000))
    return tuple(stats.median(c) for c in zip(*rows))


# Sans prélude : `joblib.load` tire l'import de sklearn -> mesure le coût réel de predict.py.
# Avec prélude : sklearn est déjà en mémoire -> isole le modèle seul (mémoire) et la désérialisation (temps).
rss_imports, rss_prod, t_load_ms = probe(MODEL)
rss_skl, rss_skl_model, t_deser_ms = probe(MODEL, prelude=SKL)
rss_modele = rss_skl_model - rss_skl

# --- Latences ---
model = joblib.load(MODEL)
x1 = X.iloc[[0]]
model.predict_proba(x1)                                     # warm-up (allocation numpy)
runs = []
for _ in range(200):
    t0 = time.perf_counter(); model.predict_proba(x1); runs.append((time.perf_counter() - t0) * 1000)
lat_1 = stats.median(runs)

batch = []
for _ in range(5):
    t0 = time.perf_counter(); model.predict_proba(X); batch.append((time.perf_counter() - t0) * 1000)
lat_batch = stats.median(batch)

# --- Latence réelle de production : 1 process python complet par patient ---
cold = []
for _ in range(5):
    t0 = time.perf_counter()
    subprocess.run([sys.executable, "legacy/predict.py", "70", "3", "28.5", "1"],
                   cwd=ROOT, capture_output=True)
    cold.append((time.perf_counter() - t0) * 1000)
lat_cold = stats.median(cold)

# --- Temps de ré-entraînement (mêmes hyperparamètres que legacy/train.py) ---
fits = []
for _ in range(3):
    t0 = time.perf_counter()
    RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0).fit(X, y)
    fits.append(time.perf_counter() - t0)
t_fit = stats.median(fits)

size_mo = MODEL.stat().st_size / 1e6
n_leaves = sum(e.get_n_leaves() for e in model.estimators_)

print(f"""
Taille artefact              : {size_mo:.2f} Mo ({MODEL.stat().st_size} o) — {n_leaves} feuilles

MÉMOIRE (RSS, process neuf, médiane sur 5 runs)
  après imports predict.py   : {rss_imports:.1f} Mo   (python + pandas + joblib)
  + import sklearn           : {rss_skl:.1f} Mo   -> sklearn seul : +{rss_skl - rss_imports:.1f} Mo
  + le modèle                : {rss_skl_model:.1f} Mo   -> MODÈLE SEUL : +{rss_modele:.1f} Mo ({rss_modele / size_mo:.1f}x son fichier)
  RSS total d'un appel prod  : {rss_prod:.1f} Mo   (dont {(1 - rss_modele / rss_prod) * 100:.0f} % = runtime, pas le modèle)

TEMPS
  joblib.load (process neuf) : {t_load_ms:.0f} ms  dont désérialisation pure : {t_deser_ms:.0f} ms
                               -> import sklearn tiré par le load : {t_load_ms - t_deser_ms:.0f} ms
  Fit (10 000 lignes)        : {t_fit * 1000:.0f} ms
  Inférence 1 ligne (chaud)  : {lat_1:.2f} ms     -> {1000 / lat_1:.0f} pred/s
  Inférence batch 10 000     : {lat_batch:.1f} ms -> {10_000 / (lat_batch / 1000):,.0f} pred/s ({lat_batch * 1000 / 10_000:.1f} us/pred)
  Appel prod (predict.py)    : {lat_cold:.0f} ms  -> {1000 / lat_cold:.2f} pred/s
      calcul utile           : {lat_1 / lat_cold * 100:.2f} %
      chargement du modèle   : {t_load_ms / lat_cold * 100:.1f} %  (dont {t_deser_ms / lat_cold * 100:.1f} % de désérialisation)
      démarrage + imports    : {(lat_cold - lat_1 - t_load_ms) / lat_cold * 100:.1f} %
  10 000 prédictions         : {lat_batch / 1000:.2f} s en batch  vs  {lat_cold * 10_000 / 3.6e6:.2f} h en mode actuel (x{lat_cold * 10_000 / lat_batch:,.0f})
""")

LEGACY = {"nom": "legacy_rf", "taille_mo": size_mo, "rss_modele_mo": rss_modele,
          "t_load_ms": t_load_ms, "t_deser_ms": t_deser_ms, "t_fit_s": t_fit,
          "lat_1ligne_ms": lat_1, "lat_batch10k_ms": lat_batch, "lat_prod_ms": lat_cold}


Windows AMD64 | Python 3.11.15 | sklearn 1.5.1 | 10 cœurs physiques / 12 logiques | 34.0 Go RAM

Taille artefact              : 4.96 Mo (4956361 o) — 30846 feuilles

MÉMOIRE (RSS, process neuf, médiane sur 5 runs)
  après imports predict.py   : 72.7 Mo   (python + pandas + joblib)
  + import sklearn           : 141.5 Mo   -> sklearn seul : +68.8 Mo
  + le modèle                : 148.5 Mo   -> MODÈLE SEUL : +7.1 Mo (1.4x son fichier)
  RSS total d'un appel prod  : 148.6 Mo   (dont 95 % = runtime, pas le modèle)

TEMPS
  joblib.load (process neuf) : 918 ms  dont désérialisation pure : 15 ms
                               -> import sklearn tiré par le load : 903 ms
  Fit (10 000 lignes)        : 337 ms
  Inférence 1 ligne (chaud)  : 1.51 ms     -> 664 pred/s
  Inférence batch 10 000     : 52.0 ms -> 192,482 pred/s (5.2 us/pred)
  Appel prod (predict.py)    : 1890 ms  -> 0.53 pred/s
      calcul utile           : 0.08 %
      chargement du modèle   : 48.6 %  (dont 0.8 % de désérialisation)

## 3. Comparaison à 2 alternatives

`logreg` (régression logistique) et `histgb` (boosting), plus la variante `logreg_sans_sexe` qui sert
de contrefactuel au volet éthique. Protocole identique pour tous : mêmes données, mêmes folds, même sonde RSS.


In [9]:
"""Comparaison sobriété : legacy vs alternatives légères, à données et protocole identiques."""
import tempfile

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
TMP = Path(tempfile.mkdtemp())
Y_REF = (df["dms_jours"] >= 5.6).astype(int).values   # règle d'étiquetage reconstruite (cf. 01_ethique § 2.3)
EST_F = (df["sexe"] == "F").values


def equite(pred, proba):
    """DI F/M sur les prédictions + FNR par sexe mesuré contre la référence y_ref."""
    di = pred[EST_F].mean() / pred[~EST_F].mean()
    fnr_f = 1 - pred[EST_F & (Y_REF == 1)].mean()
    fnr_m = 1 - pred[~EST_F & (Y_REF == 1)].mean()
    return di, fnr_f, fnr_m, proba[EST_F].mean(), proba[~EST_F].mean()


def mesurer(nom, est, Xi):
    """Fit + taille + RSS modèle seul + latences + qualité CV + équité — protocole identique pour tous."""
    fits = []
    for _ in range(3):
        t0 = time.perf_counter(); est.fit(Xi, y); fits.append(time.perf_counter() - t0)

    path = TMP / f"{nom}.joblib"
    joblib.dump(est, path)
    rss_b, rss_a, t_deser = probe(path, prelude=SKL, n=3)   # sklearn préchargé -> RSS du modèle seul

    xi1 = Xi.iloc[[0]]
    est.predict_proba(xi1)                                   # warm-up
    r = [(lambda t0: (est.predict_proba(xi1), (time.perf_counter() - t0) * 1000)[1])(time.perf_counter())
         for _ in range(200)]
    b = [(lambda t0: (est.predict_proba(Xi), (time.perf_counter() - t0) * 1000)[1])(time.perf_counter())
         for _ in range(5)]

    sc = cross_validate(est, Xi, y, cv=CV, scoring=["accuracy", "f1", "roc_auc"])
    proba = est.predict_proba(Xi)[:, 1]
    di, fnr_f, fnr_m, p_f, p_m = equite((proba >= 0.5).astype(int), proba)
    return {"nom": nom, "t_fit_s": stats.median(fits), "taille_ko": path.stat().st_size / 1e3,
            "rss_modele_mo": rss_a - rss_b, "t_deser_ms": t_deser,
            "lat1_ms": stats.median(r), "latb_ms": stats.median(b),
            "acc": sc["test_accuracy"].mean(), "f1": sc["test_f1"].mean(), "auc": sc["test_roc_auc"].mean(),
            "di": di, "fnr_f": fnr_f, "fnr_m": fnr_m, "p_f": p_f, "p_m": p_m}


X_ns = X.drop(columns=["sexe_bin"])                          # variante : mêmes features, sans le sexe
logreg = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

R = pd.DataFrame([
    mesurer("legacy_rf", RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0), X),
    mesurer("logreg", logreg(), X),
    mesurer("histgb", HistGradientBoostingClassifier(max_iter=100, random_state=0), X),
    mesurer("logreg_sans_sexe", logreg(), X_ns),
]).set_index("nom")

ref = R.loc["legacy_rf"]
R["taille_x"] = (ref.taille_ko / R.taille_ko).round(0)
R["fit_x"] = (ref.t_fit_s / R.t_fit_s).round(1)
R["rss_x"] = (ref.rss_modele_mo / R.rss_modele_mo).round(1)
R["latb_x"] = (ref.latb_ms / R.latb_ms).round(1)

pd.set_option("display.width", 220, "display.max_columns", 30)
print(f"Banc : {BANC}\n")
print("=== Ressources (legacy_rf = référence) ===")
print(R[["taille_ko", "rss_modele_mo", "t_deser_ms", "t_fit_s", "lat1_ms", "latb_ms",
         "taille_x", "fit_x", "rss_x", "latb_x"]].round(3).to_string())
print("\n=== Qualité (CV 5 folds stratifiés, shuffle, random_state=42) ===")
print(R[["acc", "f1", "auc"]].round(4).to_string())
print("\n=== Équité (in-sample, même protocole que 01_ethique — DI cible ≥ 0,80) ===")
print(R[["di", "fnr_f", "fnr_m", "p_f", "p_m"]].round(4).to_string())


Banc : Windows AMD64 | Python 3.11.15 | sklearn 1.5.1 | 10 cœurs physiques / 12 logiques | 34.0 Go RAM

=== Ressources (legacy_rf = référence) ===
                  taille_ko  rss_modele_mo  t_deser_ms  t_fit_s  lat1_ms  latb_ms  taille_x  fit_x  rss_x  latb_x
nom                                                                                                              
legacy_rf          4959.529          7.053      24.666    0.338    1.524   44.476       1.0    1.0    1.0     1.0
logreg                1.822          0.025       1.533    0.010    1.260    3.670    2722.0   33.0  287.0    12.1
histgb              366.080          0.737       9.013    0.508    2.630   21.786      14.0    0.7    9.6     2.0
logreg_sans_sexe      1.758          0.029       3.285    0.007    0.389    2.859    2821.0   50.6  246.0    15.6

=== Qualité (CV 5 folds stratifiés, shuffle, random_state=42) ===
                     acc      f1     auc
nom                                     
legacy_rf         0.

## 4. Coût du modèle — ordre de grandeur

Toutes les hypothèses sont **explicites et paramétrables** ci-dessous. L'objectif n'est pas un chiffrage
comptable mais un **ordre de grandeur** permettant de comparer les postes entre eux.


In [10]:
"""Coût du modèle — ordre de grandeur, hypothèses explicites (à valider avec Hélène : Q_vol, Q_geste)."""
# --- Hypothèses (à confirmer par MediVox) ---
H = {
    "sejours_an": 10_000,      # = taille du dataset ; volume réel INCONNU -> Q16
    "geste_min": 2,            # minutes humaines par appel SSH (connexion, saisie, lecture, report) -> Q17
    "cout_horaire_eur": 45,    # coût chargé d'un profil soignant/administratif
    "puissance_w": 25,         # puissance moyenne d'un cœur x86 serveur sous charge
    "prix_kwh_eur": 0.174,     # tarif pro France 2025
    "kgco2_kwh": 0.056,        # mix électrique français (ordre de grandeur ADEME/RTE)
    "serveur_eur_an": 600,     # VM ou amortissement serveur dédié, 24/7
}

n = H["sejours_an"]
cpu_h_actuel = lat_cold * n / 3.6e6                       # 1 process python par patient
cpu_h_batch = (lat_batch / 1000 + lat_cold / 1000) / 3600  # 1 seul démarrage + 1 batch
kwh = cpu_h_actuel * H["puissance_w"] / 1000
h_humaines = n * H["geste_min"] / 60

postes = pd.DataFrame([
    ("Calcul — mode actuel (1 process/patient)", cpu_h_actuel * H["puissance_w"] / 1000 * H["prix_kwh_eur"]),
    ("Calcul — même volume en batch", cpu_h_batch * H["puissance_w"] / 1000 * H["prix_kwh_eur"]),
    ("Ré-entraînement (1x/an)", t_fit * H["puissance_w"] / 3.6e6 * H["prix_kwh_eur"]),
    ("Serveur dédié 24/7 (coût fixe)", H["serveur_eur_an"]),
    ("Geste humain SSH", h_humaines * H["cout_horaire_eur"]),
], columns=["poste", "eur_an"]).set_index("poste")
postes["part_%"] = (postes.eur_an / postes.eur_an.sum() * 100).round(2)

print(f"""HYPOTHÈSES : {n:,} séjours/an · {H['geste_min']} min de geste humain/appel · {H['cout_horaire_eur']} €/h
             {H['puissance_w']} W · {H['prix_kwh_eur']} €/kWh · {H['kgco2_kwh']} kgCO2e/kWh · serveur {H['serveur_eur_an']} €/an

CALCUL
  CPU mode actuel   : {cpu_h_actuel:.2f} h/an  -> {kwh:.3f} kWh  -> {kwh * H['kgco2_kwh'] * 1000:.0f} gCO2e/an
  CPU en batch      : {cpu_h_batch * 3600:.1f} s/an ({cpu_h_actuel * 3600 / (cpu_h_batch * 3600):,.0f}x moins)
  Taux d'occupation du serveur : {cpu_h_actuel / 8760 * 100:.3f} % de l'année
  Ré-entraînement   : {t_fit:.2f} s de CPU, 1 fois -> coût électrique {t_fit * H['puissance_w'] / 3.6e6 * H['prix_kwh_eur']:.2e} €

HUMAIN
  Geste SSH         : {h_humaines:,.0f} h/an, soit {h_humaines / 1607:.1f} ETP

POSTES (€/an, ordre de grandeur)""")
print(postes.round(2).to_string())
print(f"""
LECTURE : le calcul coûte {postes.loc['Calcul — mode actuel (1 process/patient)', 'eur_an']:.2f} €/an.
Le geste humain coûte {postes.loc['Geste humain SSH', 'eur_an'] / max(postes.loc['Calcul — mode actuel (1 process/patient)', 'eur_an'], 1e-9):,.0f}x plus cher.
Empreinte carbone annuelle du calcul : {kwh * H['kgco2_kwh'] * 1000:.0f} gCO2e — soit ~{kwh * H['kgco2_kwh'] / 0.103 * 1000:.0f} m parcourus en voiture thermique.
Avec logreg, l'artefact passerait de {size_mo * 1000:.0f} Ko à {R.loc['logreg', 'taille_ko']:.1f} Ko : gain réel = 0 € — le modèle n'a JAMAIS été le coût.
""")


HYPOTHÈSES : 10,000 séjours/an · 2 min de geste humain/appel · 45 €/h
             25 W · 0.174 €/kWh · 0.056 kgCO2e/kWh · serveur 600 €/an

CALCUL
  CPU mode actuel   : 5.25 h/an  -> 0.131 kWh  -> 7 gCO2e/an
  CPU en batch      : 1.9 s/an (9,733x moins)
  Taux d'occupation du serveur : 0.060 % de l'année
  Ré-entraînement   : 0.34 s de CPU, 1 fois -> coût électrique 4.07e-07 €

HUMAIN
  Geste SSH         : 333 h/an, soit 0.2 ETP

POSTES (€/an, ordre de grandeur)
                                            eur_an  part_%
poste                                                     
Calcul — mode actuel (1 process/patient)      0.02    0.00
Calcul — même volume en batch                 0.00    0.00
Ré-entraînement (1x/an)                       0.00    0.00
Serveur dédié 24/7 (coût fixe)              600.00    3.85
Geste humain SSH                          15000.00   96.15

LECTURE : le calcul coûte 0.02 €/an.
Le geste humain coûte 656,644x plus cher.
Empreinte carbone annuelle du calcul : 

---

# 5. Volet technique — justification des chiffres de `audit/02_technique.md`

Chaque sous-section produit les mesures citées dans le volet technique. Les cellules **réutilisent
les objets de la section 2** (`ROOT`, `MODEL`, `DATA`, `df`, `X`, `y`, `model`, `probe`, `SKL`) et
de la section 3 (`CV`) : exécuter le notebook dans l'ordre.

> ⚠️ Aucune cellule ne modifie le repo de façon durable. La seule qui écrit
> (`5.3`, qui exécute `legacy/train.py`) **sauvegarde puis restaure** l'artefact, avec vérification
> du hash en `finally`.


## 5.1 Architecture, couplage et versionning (§ 3.1 – 3.3)

Justifie : `def`/`class` = 0, 25 lignes de code utile, 3 chemins relatifs codés en dur, échec en
code retour 2 hors de la racine, 2 commits / 0 tag, binaire de 4,73 Mio versionné, secret présent
dans l'historique Git.


In [12]:
"""§3.1-3.3 — inventaire statique du legacy : modularité, chemins en dur, couplage cwd, versionning."""
import ast
import hashlib
import re

FICHIERS = [ROOT / "legacy" / "train.py", ROOT / "legacy" / "predict.py", ROOT / "tests" / "test_smoke.py"]


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()


def inventaire(path):
    src = path.read_text(encoding="utf-8")
    arbre = ast.parse(src)
    noeuds = list(ast.walk(arbre))
    lignes = src.splitlines()
    # littéraux ressemblant à un chemin relatif : c'est eux qui imposent le cwd
    chemins = [n.value for n in noeuds
               if isinstance(n, ast.Constant) and isinstance(n.value, str) and re.match(r"^[\w.]+/", n.value)]
    return {"fichier": path.name,
            "lignes": len(lignes),
            "code": sum(1 for l in lignes if l.strip() and not l.strip().startswith("#")),
            "def": sum(isinstance(n, ast.FunctionDef) for n in noeuds),
            "class": sum(isinstance(n, ast.ClassDef) for n in noeuds),
            "try": sum(isinstance(n, ast.Try) for n in noeuds),
            "logging": src.count("logging"),
            "chemins_en_dur": len(chemins),
            "detail_chemins": " ; ".join(chemins)}


INV = pd.DataFrame([inventaire(f) for f in FICHIERS]).set_index("fichier")
print("=== Modularité et chemins en dur ===")
print(INV.drop(columns="detail_chemins").to_string())
for f, c in INV.detail_chemins.items():
    if c:
        print(f"  {f:16} -> {c}")
print(f"\nCode utile du legacy : {INV.loc[['train.py', 'predict.py'], 'code'].sum()} lignes, "
      f"{INV.loc[['train.py', 'predict.py'], 'def'].sum()} fonction(s), "
      f"{INV.loc[['train.py', 'predict.py'], 'class'].sum()} classe(s)")

# --- Couplage au répertoire courant : le même appel, depuis ailleurs ---
print("\n=== Couplage au cwd (même commande, deux répertoires) ===")
for cwd in (ROOT, ROOT.parent):
    out = subprocess.run([sys.executable, "legacy/predict.py", "70", "3", "28.5", "1"],
                         cwd=cwd, capture_output=True, text=True)
    msg = out.stdout.strip() or out.stderr.strip().splitlines()[-1]
    print(f"  cwd={str(cwd):<70} rc={out.returncode}  {msg[:90]}")

# --- Versionning : poids mesuré sur les fichiers réellement suivis par Git ---
suivis = pd.Series({f: (ROOT / f).stat().st_size for f in git("ls-files").splitlines()
                    if (ROOT / f).exists()}).sort_values(ascending=False)
taille_modele = MODEL.stat().st_size
print(f"""
=== Versionning ===
  commits                    : {git("rev-list", "--count", "HEAD")}
  tags                       : {git("tag") or "AUCUN"}
  fichiers suivis par Git    : {len(suivis)} pour {suivis.sum() / 1024**2:.2f} Mio
  modèle suivi par Git       : {git("ls-files", "legacy/dms_predictor_v1.joblib") or "non"}  ({taille_modele} o = {taille_modele / 1024**2:.2f} Mio)
  part du modèle dans le suivi: {taille_modele / suivis.sum() * 100:.1f} % du volume versionné
  données suivies par Git    : {git("ls-files", "data/dms_dataset.csv") or "non"}  ({DATA.stat().st_size} o)
  .gitignore                 : {(ROOT / ".gitignore").read_text(encoding="utf-8").split() or "vide"}
  sha256 du modèle livré     : {sha256(MODEL)}

  Top 3 des fichiers versionnés :""")
print(suivis.head(3).apply(lambda v: f"{v / 1024:.0f} Kio").to_string())

print("\n=== Secret dans l'historique Git (§ 3.4.1) ===")
fuite = git("log", "--all", "-S", "DB_PASSWORD", "--oneline")
print(f"  commits introduisant/modifiant DB_PASSWORD :\n{fuite or '  aucun'}")
print("  -> supprimer la ligne de train.py ne retire PAS le secret de ces commits")


=== Modularité et chemins en dur ===
               lignes  code  def  class  try  logging  chemins_en_dur
fichier                                                              
train.py           26    13    0      0    0        0               2
predict.py         20    12    0      0    0        0               1
test_smoke.py      53    41    3      0    0        0               1
  train.py         -> data/dms_dataset.csv ; legacy/dms_predictor_v1.joblib
  predict.py       -> legacy/dms_predictor_v1.joblib
  test_smoke.py    -> legacy/predict.py

Code utile du legacy : 25 lignes, 0 fonction(s), 0 classe(s)

=== Couplage au cwd (même commande, deux répertoires) ===
  cwd=c:\git\repository\formation\M7-B1-MediVox-audit                        rc=0  RISQUE_SEJOUR_PROLONGE 0.748
  cwd=c:\git\repository\formation                                            rc=2  C:\Users\A452415\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\python.exe: ca

=== Versionning ===
  commits       

## 5.2 Sécurité — validation des entrées (§ 3.4.2)

Les 11 appels réels de `legacy/predict.py` cités dans le volet technique. La colonne `verdict`
compare l'argument au **domaine d'entraînement** (bornes lues dans le dataset) : tout appel qui
sort du domaine **et** rend un code retour 0 est une décision rendue hors du périmètre du modèle,
sans aucun signal pour l'appelant.


In [13]:
"""§3.4.2 — valeurs aux limites : 11 appels réels de predict.py, confrontés au domaine d'entraînement."""
DOMAINE = {"age": (df.age.min(), df.age.max()),
           "nb_comorbidites": (df.nb_comorbidites.min(), df.nb_comorbidites.max()),
           "imc": (df.imc.min(), df.imc.max()),
           "sexe_bin": (0, 1)}
print("Domaine d'entraînement :", {k: (float(a), float(b)) for k, (a, b) in DOMAINE.items()}, "\n")

CAS = {
    "nominal": ["70", "3", "28.5", "1"],
    "age négatif": ["-5", "0", "22", "0"],
    "age = 500 ans": ["500", "0", "22", "0"],
    "imc = 0": ["70", "3", "0", "1"],
    "imc = 900": ["70", "3", "900", "1"],
    "99 comorbidités": ["70", "99", "28.5", "1"],
    "sexe_bin = 7": ["70", "3", "28.5", "7"],
    "argument texte": ["abc", "3", "28.5", "1"],
    "argument manquant": ["70", "3", "28.5"],
    "argument en trop": ["70", "3", "28.5", "1", "666"],
    "aucun argument": [],
}


def hors_domaine(args):
    """Nombre d'arguments hors des bornes vues à l'entraînement (les non numériques ne comptent pas)."""
    n = 0
    for val, (lo, hi) in zip(args[:4], DOMAINE.values()):
        try:
            n += not (lo <= float(val) <= hi)
        except ValueError:
            pass
    return n


def appel(args):
    out = subprocess.run([sys.executable, "legacy/predict.py", *args], cwd=ROOT, capture_output=True, text=True)
    err = out.stderr.strip().splitlines()[-1] if out.stderr.strip() else ""
    return {"args": " ".join(args) or "(vide)", "rc": out.returncode,
            "sortie": out.stdout.strip() or err[:60], "hors_domaine": hors_domaine(args)}


LIMITES = pd.DataFrame([{"cas": k, **appel(v)} for k, v in CAS.items()]).set_index("cas")
LIMITES["verdict"] = [
    "OK" if (hd == 0 and rc == 0) else
    "ACCEPTÉ HORS DOMAINE" if (hd > 0 and rc == 0) else
    "crash brut"
    for hd, rc in zip(LIMITES.hors_domaine, LIMITES.rc)
]
print(LIMITES.to_string())

accepte_faux = (LIMITES.verdict == "ACCEPTÉ HORS DOMAINE").sum()
print(f"""
BILAN : {accepte_faux} appel(s) sur {len(LIMITES)} portent des valeurs hors du domaine
        d'entraînement et rendent malgré tout une décision avec code retour 0.
        {(LIMITES.rc != 0).sum()} échec(s) sortent en traceback sur stderr — non journalisé (§ 3.6).
        Aucun `try`, aucun schéma, aucune borne dans predict.py (cf. inventaire 5.1).""")


Domaine d'entraînement : {'age': (18.0, 94.0), 'nb_comorbidites': (0.0, 7.0), 'imc': (15.0, 44.1), 'sexe_bin': (0.0, 1.0)} 

                              args  rc                                                     sortie  hors_domaine               verdict
cas                                                                                                                                  
nominal                70 3 28.5 1   0                               RISQUE_SEJOUR_PROLONGE 0.748             0                    OK
age négatif              -5 0 22 0   0                                      SEJOUR_STANDARD 0.097             1  ACCEPTÉ HORS DOMAINE
age = 500 ans           500 0 22 0   0                                      SEJOUR_STANDARD 0.261             1  ACCEPTÉ HORS DOMAINE
imc = 0                   70 3 0 1   0                               RISQUE_SEJOUR_PROLONGE 0.837             1  ACCEPTÉ HORS DOMAINE
imc = 900               70 3 900 1   0                               RI

## 5.3 Reproductibilité — l'artefact livré est-il celui que le repo régénère ? (§ 3.5)

Deux questions distinctes :

1. **`train.py` est-il déterministe ?** → ré-entraînement deux fois de suite, mêmes hyperparamètres.
2. **L'artefact livré est-il le même que celui régénéré ?** → comparaison des prédictions et du hash.

> ⚠️ Cette cellule exécute `legacy/train.py`, qui **écrase** `legacy/dms_predictor_v1.joblib`.
> L'artefact est sauvegardé avant et **restauré dans un `finally`**, avec vérification du hash.


In [14]:
"""§3.5 — reproductibilité : déterminisme de train.py, puis artefact livré vs artefact régénéré."""
import shutil

# --- 1. train.py est-il déterministe ? (deux fits successifs, mêmes hyperparamètres) ---
a = RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0).fit(X, y)
b = RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0).fit(X, y)
print(f"Déterminisme sur cette machine : {(a.predict(X) == b.predict(X)).mean() * 100:.2f} % de prédictions identiques")

# --- 2. L'artefact LIVRÉ est-il celui que ce code régénère ? ---
livre, regen = model.predict(X), a.predict(X)
pr_livre, pr_regen = model.predict_proba(X)[:, 1], a.predict_proba(X)[:, 1]
desaccord = (livre != regen).mean()
print(f"""
Artefact livré vs régénéré (mêmes données, même random_state, même machine)
  désaccord sur les décisions : {desaccord * 100:.2f} %  ({int(desaccord * len(X))} séjours sur {len(X)})
  écart de probabilité        : moyen {abs(pr_livre - pr_regen).mean():.4f} | max {abs(pr_livre - pr_regen).max():.4f}
  accuracy TRAIN livré        : {(livre == y).mean():.4f}
  accuracy TRAIN régénéré     : {(regen == y).mean():.4f}""")

# --- 3. Hash de l'artefact après exécution réelle de legacy/train.py (avec restauration garantie) ---
h_livre = sha256(MODEL)
sauvegarde = TMP / "dms_predictor_v1.livre.joblib"
shutil.copy2(MODEL, sauvegarde)
try:
    out = subprocess.run([sys.executable, "legacy/train.py"], cwd=ROOT, capture_output=True, text=True, check=True)
    h_regen = sha256(MODEL)
    print(f"""
  `python legacy/train.py` -> {out.stdout.strip()}
  sha256 livré    : {h_livre}
  sha256 régénéré : {h_regen}
  binaires identiques : {h_livre == h_regen}""")
finally:
    shutil.copy2(sauvegarde, MODEL)
    assert sha256(MODEL) == h_livre, "RESTAURATION ÉCHOUÉE — restaurer via `git checkout legacy/`"
    print(f"  artefact d'origine restauré : OK ({sha256(MODEL)[:12]}…)")

print(f"""
LECTURE : `train.py` est déterministe *sur une machine donnée*, mais l'artefact livré
diverge de celui que le repo régénère ({desaccord * 100:.2f} % des décisions). Il a donc été produit avec
une autre version de bibliothèque, une autre plateforme ou un autre jeu de données —
et rien dans le repo ne dit lequel : aucune métadonnée n'accompagne le modèle (§ 3.3).
-> l'audit porte sur un modèle PROCHE de celui de production, pas sur lui (Q8).""")


Déterminisme sur cette machine : 100.00 % de prédictions identiques

Artefact livré vs régénéré (mêmes données, même random_state, même machine)
  désaccord sur les décisions : 0.16 %  (16 séjours sur 10000)
  écart de probabilité        : moyen 0.0013 | max 0.0233
  accuracy TRAIN livré        : 0.7721
  accuracy TRAIN régénéré     : 0.7715

  `python legacy/train.py` -> modele entraine et sauve. accuracy train = 0.7715
  sha256 livré    : 33408e6aacacf79648c806ac53037581129b7d6433f42237a2cd223f5a14ec4a
  sha256 régénéré : 0061d7d84f5a7da665ccfdf08c6bce19a3f38ca0dae3649cca8c4d5eb0e86a40
  binaires identiques : False
  artefact d'origine restauré : OK (33408e6aacac…)

LECTURE : `train.py` est déterministe *sur une machine donnée*, mais l'artefact livré
diverge de celui que le repo régénère (0.16 % des décisions). Il a donc été produit avec
une autre version de bibliothèque, une autre plateforme ou un autre jeu de données —
et rien dans le repo ne dit lequel : aucune métadonnée n'accomp

## 5.4 Qualité réelle du modèle et utilité des features (§ 3.7)

`train.py` imprime une accuracy **mesurée sur les données d'entraînement**, sans split : c'est le
seul chiffre de performance que MediVox possède. On le confronte ici à une validation croisée, à
la baseline majoritaire, puis on mesure ce que chaque feature apporte réellement (ablation).


In [15]:
"""§3.7 — performance annoncée vs réelle, valeur de chaque feature, capacité du modèle."""
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score

pr = model.predict_proba(X)[:, 1]
acc_train, auc_train = accuracy_score(y, pr >= 0.5), roc_auc_score(y, pr)


def rf():
    return RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0)


def cv(est, Xi):
    return (cross_val_score(est, Xi, y, cv=CV, scoring="accuracy"),
            cross_val_score(est, Xi, y, cv=CV, scoring="roc_auc"))


acc_cv, auc_cv = cv(rf(), X)
acc_dummy, _ = cv(DummyClassifier(strategy="most_frequent"), X)

print(f"""=== Performance annoncée vs performance réelle ===
  accuracy TRAIN (chiffre imprimé par train.py) : {acc_train:.4f}
  accuracy CV 5 folds                           : {acc_cv.mean():.4f} +/- {acc_cv.std():.4f}
  -> sur-apprentissage                          : {(acc_train - acc_cv.mean()) * 100:.1f} points

  AUC TRAIN                                     : {auc_train:.4f}
  AUC CV 5 folds                                : {auc_cv.mean():.4f} +/- {auc_cv.std():.4f}
  -> sur-apprentissage                          : {(auc_train - auc_cv.mean()) * 100:.1f} points

  baseline "tout le monde = séjour standard"    : {acc_dummy.mean():.4f}
  -> gain réel apporté par le modèle            : {(acc_cv.mean() - acc_dummy.mean()) * 100:.1f} points""")

# --- Ablation : ce que chaque feature apporte vraiment, face à l'importance qu'elle reçoit ---
ablations = [("4 features (modèle actuel)", X)] + [(f"sans `{c}`", X.drop(columns=[c])) for c in X.columns]
ABL = pd.DataFrame([{"configuration": nom,
                     "acc_cv": cv(rf(), Xi)[0].mean(),
                     "auc_cv": cv(rf(), Xi)[1].mean()} for nom, Xi in ablations]).set_index("configuration")
ABL["delta_acc_pts"] = (ABL.acc_cv - ABL.loc["4 features (modèle actuel)", "acc_cv"]) * 100
ABL["importance_%"] = [None] + list((model.feature_importances_ * 100).round(2))
print("\n=== Ablation : utilité réelle vs importance déclarée ===")
print(ABL.round(4).to_string())
print("  -> une feature très 'importante' mais dont le retrait ne coûte rien est du bruit appris.")

# --- Capacité : le modèle a-t-il la place de mémoriser ses patients ? ---
feuilles = sum(e.get_n_leaves() for e in model.estimators_)
noeuds = sum(e.tree_.node_count for e in model.estimators_)
profondeurs = [e.get_depth() for e in model.estimators_]
print(f"""
=== Capacité vs volume de données ===
  arbres / nœuds / feuilles   : {model.n_estimators} / {noeuds} / {feuilles}
  profondeur (min-max)        : {min(profondeurs)}-{max(profondeurs)} (limite max_depth={model.max_depth})
  échantillons d'entraînement : {len(X)}
  feuilles par échantillon    : {feuilles / len(X):.2f}
  taille artefact / données   : {MODEL.stat().st_size / DATA.stat().st_size:.1f}x
  min_samples_leaf            : {model.min_samples_leaf} (réglage le plus permissif possible)
  class_weight                : {model.class_weight} — cible déséquilibrée {y.value_counts(normalize=True).round(3).to_dict()}""")


=== Performance annoncée vs performance réelle ===
  accuracy TRAIN (chiffre imprimé par train.py) : 0.7721
  accuracy CV 5 folds                           : 0.6835 +/- 0.0123
  -> sur-apprentissage                          : 8.9 points

  AUC TRAIN                                     : 0.8656
  AUC CV 5 folds                                : 0.7245 +/- 0.0113
  -> sur-apprentissage                          : 14.1 points

  baseline "tout le monde = séjour standard"    : 0.5941
  -> gain réel apporté par le modèle            : 8.9 points

=== Ablation : utilité réelle vs importance déclarée ===
                            acc_cv  auc_cv  delta_acc_pts  importance_%
configuration                                                          
4 features (modèle actuel)  0.6835  0.7245           0.00           NaN
sans `age`                  0.6537  0.6687          -2.98         35.20
sans `nb_comorbidites`      0.6396  0.6471          -4.39         25.07
sans `imc`                  0.6787  0.

## 5.5 Stabilité de la décision (§ 3.7, fin)

`predict.py` l.20 imprime une **décision binaire** obtenue par un seuil `0.5` codé en dur. Deux
mesures de fragilité : combien de patients sont assis sur ce seuil, et combien changent de
décision si l'on ne modifie **que** la variable sexe (contrefactuel individuel du § 2.4).


In [17]:
"""§3.7 — fragilité du seuil 0.5 et contrefactuel « même patient, sexe inversé »."""
SEUIL = 0.5  # valeur codée en dur dans predict.py l.20


def esp(n):
    return f"{int(n):,}".replace(",", " ")


print("=== Patients assis sur le seuil ===")
for demi in (0.05, 0.10):
    dans = ((pr >= SEUIL - demi) & (pr <= SEUIL + demi)).mean()
    print(f"  probabilité dans [{SEUIL - demi:.2f} ; {SEUIL + demi:.2f}] : {dans * 100:5.2f} %  "
          f"({esp(dans * len(X))} séjours) -> décision retournée par +/-{demi:.2f}")

# --- Contrefactuel : on ne change QUE le sexe ---
X_flip = X.copy()
X_flip["sexe_bin"] = 1 - X_flip["sexe_bin"]
pr_flip = model.predict_proba(X_flip)[:, 1]
bascule = (pr >= SEUIL) != (pr_flip >= SEUIL)

print(f"""
=== Contrefactuel « même patient, sexe inversé » ===
  décisions inversées        : {bascule.mean() * 100:.2f} %  ({esp(bascule.sum())} séjours sur {esp(len(X))})
  écart de probabilité moyen : {abs(pr - pr_flip).mean():.4f}
  sens de la bascule         : F->M {esp((bascule & (X.sexe_bin == 0)).sum())} | M->F {esp((bascule & (X.sexe_bin == 1)).sum())}
  proba moyenne codée F      : {pr[X.sexe_bin == 0].mean():.3f}   codée M : {pr[X.sexe_bin == 1].mean():.3f}

LECTURE : à données cliniques strictement identiques, le seul basculement de `sexe_bin`
inverse la décision pour {bascule.mean() * 100:.1f} % des séjours. C'est la traduction individuelle du
disparate impact mesuré en section 1, et la raison pour laquelle une décision binaire
issue d'un seuil non paramétrable (AUC CV {auc_cv.mean():.2f}) n'a pas la précision qu'elle affiche.""")


=== Patients assis sur le seuil ===
  probabilité dans [0.45 ; 0.55] : 12.03 %  (1 203 séjours) -> décision retournée par +/-0.05
  probabilité dans [0.40 ; 0.60] : 24.02 %  (2 402 séjours) -> décision retournée par +/-0.10

=== Contrefactuel « même patient, sexe inversé » ===
  décisions inversées        : 37.05 %  (3 705 séjours sur 10 000)
  écart de probabilité moyen : 0.1898
  sens de la bascule         : F->M 1 845 | M->F 1 860
  proba moyenne codée F      : 0.321   codée M : 0.491

LECTURE : à données cliniques strictement identiques, le seul basculement de `sexe_bin`
inverse la décision pour 37.0 % des séjours. C'est la traduction individuelle du
disparate impact mesuré en section 1, et la raison pour laquelle une décision binaire
issue d'un seuil non paramétrable (AUC CV 0.72) n'a pas la précision qu'elle affiche.


## 5.6 Récapitulatif — les chiffres repris dans `audit/02_technique.md`

Table unique regroupant les mesures des sections 2 et 5, pour vérifier d'un coup d'œil que le
rapport et le notebook disent la même chose. Toute divergence signifie que le rapport doit être
mis à jour, **pas l'inverse**.


In [18]:
"""Récapitulatif : chaque ligne correspond à un chiffre cité dans audit/02_technique.md."""
RECAP = pd.DataFrame([
    ("3.1", "lignes de code utile du legacy", f"{INV.loc[['train.py', 'predict.py'], 'code'].sum()}"),
    ("3.1", "fonctions / classes dans le legacy", "0 / 0"),
    ("3.1", "chemins relatifs codés en dur", f"{INV.loc[['train.py', 'predict.py'], 'chemins_en_dur'].sum()}"),
    ("3.1", "code retour hors de la racine du repo", "2"),
    ("3.2", "commits / tags", f"{git('rev-list', '--count', 'HEAD')} / {len(git('tag').split())}"),
    ("3.2", "part du modèle dans le volume versionné", f"{MODEL.stat().st_size / suivis.sum() * 100:.1f} %"),
    ("3.4", "commits contenant DB_PASSWORD", f"{len(git('log', '--all', '-S', 'DB_PASSWORD', '--oneline').splitlines())}"),
    ("3.4", "appels hors domaine acceptés (rc=0)", f"{accepte_faux} / {len(LIMITES)}"),
    ("3.4", "appels finissant en traceback non journalisé", f"{(LIMITES.rc != 0).sum()} / {len(LIMITES)}"),
    ("3.5", "désaccord artefact livré vs régénéré", f"{desaccord * 100:.2f} % ({int(desaccord * len(X))} séjours)"),
    ("3.5", "binaires identiques après train.py", f"{h_livre == h_regen}"),
    ("3.7", "accuracy train (annoncée) vs CV", f"{acc_train:.4f} vs {acc_cv.mean():.4f}"),
    ("3.7", "AUC train vs CV", f"{auc_train:.4f} vs {auc_cv.mean():.4f}"),
    ("3.7", "baseline majoritaire / gain réel", f"{acc_dummy.mean():.4f} / +{(acc_cv.mean() - acc_dummy.mean()) * 100:.1f} pts"),
    ("3.7", "coût du retrait d'`imc` (importance 30,2 %)", f"{ABL.loc['sans `imc`', 'delta_acc_pts']:.2f} pt"),
    ("3.7", "coût du retrait de `sexe_bin`", f"{ABL.loc['sans `sexe_bin`', 'delta_acc_pts']:.2f} pt"),
    ("3.7", "feuilles / échantillon d'entraînement", f"{feuilles / len(X):.2f}"),
    ("3.7", "taille artefact / taille données", f"{MODEL.stat().st_size / DATA.stat().st_size:.1f}x"),
    ("3.7", "séjours dans [0,45 ; 0,55]", f"{((pr >= 0.45) & (pr <= 0.55)).mean() * 100:.2f} %"),
    ("3.7", "décisions inversées par le seul sexe", f"{bascule.mean() * 100:.2f} %"),
    ("3.8", "tests portant sur la qualité du modèle", f"0 / {INV.loc['test_smoke.py', 'def']}"),
    ("3.9", "latence d'un appel de production", f"{lat_cold:.0f} ms"),
    ("3.9", "part de calcul utile dans cet appel", f"{lat_1 / lat_cold * 100:.2f} %"),
    ("3.9", "débit prod vs batch", f"{1000 / lat_cold:.2f}/s vs {10_000 / (lat_batch / 1000):,.0f}/s"),
    ("3.9", "RSS d'un appel prod / part du modèle", f"{rss_prod:.1f} Mo / {rss_modele / rss_prod * 100:.0f} %"),
], columns=["§", "indicateur", "mesure"]).set_index("§")
print(RECAP.to_string())


                                       indicateur               mesure
§                                                                     
3.1                lignes de code utile du legacy                   25
3.1            fonctions / classes dans le legacy                0 / 0
3.1                 chemins relatifs codés en dur                    3
3.1         code retour hors de la racine du repo                    2
3.2                                commits / tags                3 / 0
3.2       part du modèle dans le volume versionné               88.0 %
3.4                 commits contenant DB_PASSWORD                    2
3.4           appels hors domaine acceptés (rc=0)               6 / 11
3.4  appels finissant en traceback non journalisé               3 / 11
3.5          désaccord artefact livré vs régénéré  0.16 % (16 séjours)
3.5            binaires identiques après train.py                False
3.7               accuracy train (annoncée) vs CV     0.7721 vs 0.6835
3.7   